In [ ]:

#  CRACK SEGMENTATION (Frames 1-15)

import cv2
import numpy as np
import os
 
input_folder = "road_member2_gaussian_contrast"
 
# Separate folder for each stage
folders_cell1 = {
    "input":         "segmentation_results/1_input_frames",
    "gray":          "segmentation_results/2_gray_frames",
    "edges":         "segmentation_results/3_edge_frames",
    "morph":         "segmentation_results/4_morph_frames",
    "contours":      "segmentation_results/5_contour_frames",
}
 
# Create all folders
for folder in folders_cell1.values():
    os.makedirs(folder, exist_ok=True)
 
for i in range(1, 16):
 
    # Load enhanced frame
    img_path = os.path.join(input_folder, f"frame_{i}.jpg")
    image = cv2.imread(img_path)
 
    if image is None:
        print(f"Frame {i} not found, skipping...")
        continue
 
    # Save input
    cv2.imwrite(os.path.join(folders_cell1["input"], f"frame_{i}.jpg"), image)
 
    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    cv2.imwrite(os.path.join(folders_cell1["gray"], f"frame_{i}.jpg"), gray)
 
    # Edge detection for crack segmentation
    edges = cv2.Canny(gray, 50, 150)
    cv2.imwrite(os.path.join(folders_cell1["edges"], f"frame_{i}.jpg"), edges)
 
    # Morphological operation to connect crack segments
    kernel = np.ones((3, 3), np.uint8)
    morph = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)
    cv2.imwrite(os.path.join(folders_cell1["morph"], f"frame_{i}.jpg"), morph)
 
    # Find contours and draw bounding boxes
    contours, _ = cv2.findContours(morph, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    output = image.copy()
 
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > 50:
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(output, (x, y), (x+w, y+h), (0, 255, 0), 2)
            cv2.putText(output, "Crack", (x, y-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
 
    cv2.imwrite(os.path.join(folders_cell1["contours"], f"frame_{i}.jpg"), output)
 
    print(f"Cell1 - Frame {i} processed and saved.")
 
print("\nCell 1 completed!")
print("Folder structure:")
for name, path in folders_cell1.items():
    print(f"  {path}/")